In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
BASELINE_MODEL_DIR = ROOT / "models" / "baseline"

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features

TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

MODEL_CLASSES = {
    "decision_tree": DecisionTreeClassifier,
    "random_forest": RandomForestClassifier,
    "xgboost": XGBClassifier,
    "linear_svm": LinearSVC,
    "mlp": MLPClassifier,
}

MODELS_TO_TRAIN = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

In [2]:
def build_pipeline(classifier):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )

In [3]:
columns_to_load = feature_columns + ["label_binary"]

for training_name, dataset_names in TRAINING_CONFIGS.items():
    training_dataframes = []

    for dataset_name in dataset_names:
        dataframe = pd.read_parquet(
            DATA_DIR / f"{dataset_name}_train.parquet",
            columns=columns_to_load,
        )
        training_dataframes.append(dataframe)

    training_df = pd.concat(training_dataframes, ignore_index=True)

    X_train = training_df[feature_columns]
    y_train = training_df["label_binary"].eq("ATTACK").astype("int8")

    for model_name in MODELS_TO_TRAIN:
        baseline_pipeline = build_pipeline( MODEL_CLASSES[model_name](random_state=1))
        baseline_pipeline.fit(X_train, y_train)

        model_dir = BASELINE_MODEL_DIR / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        model_path = model_dir / f"{training_name}.joblib"

        joblib.dump(
            baseline_pipeline,
            model_path,
            compress=3,
        )

C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
